In [ ]:
import os

# Replace with the actual address of your Prefect server
#os.environ["PREFECT_API_URL"] = "http://prefect.local/api"
#os.environ["PREFECT_API_URL"] = "http://10.43.153.8:4200/api"
#os.environ["PREFECT_API_URL"] = "http://10.2.97.207:4200/api"

In [2]:
from dask_kubernetes.operator import KubeCluster 
from dask.distributed import Client 
import dask.array as da 
import time
import os

In [3]:
import dask
import dask.distributed
from prefect_dask.task_runners import DaskTaskRunner

/opt/conda/lib/python3.12/site-packages/tzlocal/unix.py:207: UserWarning: Can not find any timezone configuration, defaulting to UTC.
  warnings.warn("Can not find any timezone configuration, defaulting to UTC.")


In [1]:
from dask_kubernetes.operator import KubeCluster, make_cluster_spec
import os

username = os.environ["JUPYTERHUB_USER"]

spec = make_cluster_spec(
    name=f"manual-dask-cluster-{username}",
    image="ghcr.io/casangi/radps-dask-worker",
    worker_command=[
        "dask-worker",
        "--nworkers", "4",
        "--nthreads", "1",
        "--memory-limit", "4G",
    ],
    scheduler_service_type="NodePort"
)

# Navigate to the worker pod template
worker_spec = spec["spec"]["worker"]["spec"]

# Add volume
worker_spec.setdefault("volumes", []).append(
    {
        "name": "shared-storage",
        "persistentVolumeClaim": {
            "claimName": "radps-hub-pvc"
        }
    }
)

# Mount volume in the worker container
worker_container = worker_spec["containers"][0]
worker_container.setdefault("volumeMounts", []).append(
    {
        "name": "shared-storage",
        "mountPath": "/home/jovyan/shared"
    }
)

# (Optional but common) permissions fix
worker_spec.setdefault("securityContext", {})["fsGroup"] = 100

cluster = KubeCluster(
    custom_cluster_spec = spec,
    namespace="radps-hub",
)

Output()

In [ ]:
# Create and scale a Dask cluster 
# This creates a DaskCluster custom resource in Kubernetes, which the 
# Dask Operator will see and use to create a scheduler and workers. 
print("Creating Dask cluster...") 
username = os.environ["JUPYTERHUB_USER"] 

# cluster = KubeCluster(name='manual-dask-cluster'+username, image="ghcr.io/casangi/radps-dask-worker", 
#                       namespace='radps-hub', scheduler_service_type="NodePort", 
#                       worker_command=["dask-worker", "--nworkers", "4", "--nthreads", "1", "--memory-limit", "4G"])

In [4]:
# Scale the cluster to 1 pod 
print("One pod with 4 workers...")
cluster.scale(2)

# It can take a minute or two for the pods to be created and ready. 
# The following call will block until the workers are available. 
cluster.wait_for_workers(2) 
print("Cluster is ready with 4 workers on 1 pod.")

One pod with 4 workers...
Cluster is ready with 4 workers on 1 pod.


In [5]:
# Connect a Dask client to the cluster 
client = Client(cluster) 
print("Dask client connected.") 
client

Dask client connected.


/opt/conda/lib/python3.12/site-packages/distributed/client.py:1590: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| toolz   | 1.0.0  | 0.12.0    | 0.12.0  |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


<Client: 'tcp://10.42.3.109:8786' processes=4 threads=4, memory=14.90 GiB>

In [ ]:
client.scheduler

In [7]:
def image_channel_chunk():
    import time
    from xradio.measurement_set import open_processing_set
    convert_out = "/home/jovyan/shared/data/Antennae_North.cal.lsrk.split.ps.zarr"
    
    ps_xdt = open_processing_set(convert_out)
    n = len(ps_xdt)
    
    time.sleep(0.1)
    return n

n_cc = 1000
return_vals_list = []
for i_cc in range(n_cc):
    delayed_return_val = dask.delayed(image_channel_chunk)()
    return_vals_list.append(delayed_return_val)
    
dask.compute(return_vals_list)

([12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12

In [8]:
client.close()

In [9]:
cluster.close()

In [ ]:
# # await run_pipeline(task_runner)

# from prefect import flow
# from prefect import task
# import dask

# @task
# def image_single_spw(x):
#     import time
#     time.sleep(1)
    
    
#     ###############################
#     def image_channel_chunk():
#         import time
#         time.sleep(0.1)
#         return 42
    
#     n_cc = 1000
#     return_vals_list = []
#     for i_cc in range(n_cc):
#         delayed_return_val = dask.delayed(image_channel_chunk)()
#         return_vals_list.append(delayed_return_val)
        
#     dask.compute(return_vals_list)
#     ###############################
        
#     return x**2

# @flow(task_runner=task_runner)
# def image_cube(n_spw):
#     futures = [image_single_spw.submit(i) for i in range(n_spw)]
#     return [f.result() for f in futures]

# image_cube(n_spw=16)

In [ ]:
task_runner=DaskTaskRunner(client.scheduler)

In [ ]:
# # await run_pipeline(task_runner)

# from prefect import flow
# from prefect import task
# import dask

# @task
# def image_single_spw(x):
#     import time
#     time.sleep(1)
    
    
#     ###############################
#     def image_channel_chunk():
#         import time
#         time.sleep(0.1)
#         return 42
    
#     n_cc = 1000
#     return_vals_list = []
#     for i_cc in range(n_cc):
#         delayed_return_val = dask.delayed(image_channel_chunk)()
#         return_vals_list.append(delayed_return_val)
        
#     dask.compute(return_vals_list)
#     ###############################
        
#     return x**2

# @flow(task_runner=task_runner)
# def image_cube(n_spw):
#     futures = [image_single_spw.submit(i) for i in range(n_spw)]
#     return [f.result() for f in futures]

# image_cube(n_spw=16)

In [ ]:
from prefect import flow, task
from prefect_dask.task_runners import DaskTaskRunner
import dask

@task
def image_channel_chunk():
    import time
    time.sleep(0.1)
    return 42

@task
def image_single_spw(x):
    import time
    time.sleep(1)

    n_cc = 1000
    return_vals_list = []
    for i_cc in range(n_cc):
        delayed_return_val = dask.delayed(image_channel_chunk.fn)()  # use .fn to get raw callable
        return_vals_list.append(delayed_return_val)
        
    dask.compute(*return_vals_list)
        
    return x**2

task_runner=DaskTaskRunner(address=client.scheduler.address)

@flow(task_runner=task_runner)
def image_cube(n_spw):
    futures = [image_single_spw.submit(i) for i in range(n_spw)]
    return [f.result() for f in futures]

image_cube(n_spw=16)

In [ ]:
str(client.scheduler.address)

In [ ]:
client.close()
cluster.close()